## ΣΥΜΦΟΡΗΣΗ (congestion)

Για να εντοπίσεις μια συμφόρηση (congestion) μετά την εκτέλεση της network.pf(), πρέπει να συγκρίνεις την ισχύ που "θέλει" να περάσει από μια γραμμή με το φυσικό της όριο (s_nom).

Το Gemini είπε

Για να εντοπίσεις μια συμφόρηση (congestion) μετά την εκτέλεση της network.pf(), πρέπει να συγκρίνεις την ισχύ που "θέλει" να περάσει από μια γραμμή με το φυσικό της όριο (s_nom).

Ακολουθεί ο τρόπος με τον οποίο μπορείς να το αναλύσεις στην εργασία σου:
#### 1. Οπτικοποίηση της Φόρτισης (Line Loading)

Η συμφόρηση συμβαίνει όταν η ροή ισχύος (P) πλησιάζει ή ξεπερνά την ονομαστική ικανότητα της γραμμής. Στο PyPSA, αυτό υπολογίζεται ως εξής:

### <p style="text-align: center;"> Loading(%)=​p|0​| / snom​ ×100 </p>

Αν το αποτέλεσμα είναι >100%, η γραμμή είναι υπερφορτωμένη.

#### 2. Ανάλυση των Γωνιών Τάσης (θ)
* Αν παρατηρήσεις μια πολύ μεγάλη διαφορά γωνίας (Δθ) ανάμεσα σε δύο ζυγούς, αυτό είναι ένδειξη ότι "πιέζεις" πάρα πολλή ενέργεια μέσα από μια διαδρομή που έχει υψηλή επαγωγική αντίδραση (X).

* Αυτό συχνά οδηγεί σε συμφόρηση πριν ακόμα φτάσουμε στο θερμικό όριο της γραμμής, λόγω προβλημάτων ευστάθειας.


In [1]:
import pypsa
import numpy as np

#Νέο Δίκτυο (Δήλωση)
net = pypsa.Network()

#Προσθέτουμε 2 Ζυγούς υψηλής τάσης 
net.add("Bus", "Bus A", v_nom=220)
net.add("Bus", "Bus B", v_nom=220)

#Συνδέουμε τους Ζυγούς μέσω γραμμής μεταφοράς
net.add("Line", "Line 1",
        bus0 = "Bus A",
        bus1 = "Bus B",
        length = 50,
        x=0.1,
        r=0.01,
        s_nom = 100)

#Προσθέτουμε μια γεννήτρια (Η/Ζ) στο ΖΥΓΟ Α
net.add("Generator", "Generator A",
        bus="Bus A",
        p_nom = 80,
        marginal_cost=50)

#Προσθέτουμε ένα φορτίο στο ΖΥΓΟ Β
net.add("Load", "Load A",
        bus="Bus B",
        p_set = 100)

#Τρέξε βασική ανάλυση ροής ρεύματος
net.pf()

net.pf()

print("\n" + "="*30)
print(" ΑΝΑΦΟΡΑ ΡΟΗΣ ΙΣΧΥΟΣ (AC) ")
print("="*30)

# 1. Αποτελέσματα Ζυγών (Τάσεις και Γωνίες)
print("\n[ ΚΑΤΑΣΤΑΣΗ ΖΥΓΩΝ ]")
for bus_name in net.buses.index:
    v_mag = net.buses_t.v_mag_pu.at["now", bus_name]
    v_ang = np.rad2deg(net.buses_t.v_ang.at["now", bus_name])
    print(f"Ζυγός {bus_name}: Τάση = {v_mag:.4f} p.u. | Γωνία = {v_ang:.2f}°")

# 2. Αποτελέσματα Γραμμών (Ροές και Φόρτιση)
print("\n[ ΡΟΕΣ ΓΡΑΜΜΩΝ ]")
for line_name in net.lines.index:
    p0 = net.lines_t.p0.at["now", line_name]
    s_nom = net.lines.at[line_name, "s_nom"]
    loading = (abs(p0) / s_nom) * 100
    print(f"Γραμμή {line_name}: Ροή = {p0:.2f} MW | Φόρτιση = {loading:.1f}%")

# 3. Πραγματική Παραγωγή (Dispatch)
print("\n[ ΠΑΡΑΓΩΓΗ ΓΕΝΝΗΤΡΙΩΝ ]")
# Υπολογίζουμε την παραγωγή από τις εγχύσεις στους ζυγούς για να αποφύγουμε το NaN
for gen_name in net.generators.index:
    bus_id = net.generators.at[gen_name, "bus"]
    # Η έγχυση στον ζυγό ισούται με την παραγωγή (αν δεν υπάρχει άλλο στοιχείο εκεί)
    actual_p = net.buses_t.p.at["now", bus_id] 
    print(f"Γεννήτρια {gen_name}: Ισχύς Εξόδου = {actual_p:.4f} MW")

# 4. Υπολογισμός Απωλειών
total_losses = net.lines_t.p0.sum().sum() - abs(net.lines_t.p1.sum().sum())
print(f"\nΣυνολικές Απώλειες Δικτύου: {abs(total_losses)*1000:.2f} kW")
print("="*30)


INFO:pypsa.network.power_flow:Performing non-linear load-flow on AC sub-network <pypsa.networks.SubNetwork object at 0x000001DBDEB9D2B0> for snapshots Index(['now'], dtype='str', name='snapshot')
INFO:pypsa.network.power_flow:Performing non-linear load-flow on AC sub-network <pypsa.networks.SubNetwork object at 0x000001DBDEAD3610> for snapshots Index(['now'], dtype='str', name='snapshot')



 ΑΝΑΦΟΡΑ ΡΟΗΣ ΙΣΧΥΟΣ (AC) 

[ ΚΑΤΑΣΤΑΣΗ ΖΥΓΩΝ ]
Ζυγός Bus A: Τάση = 1.0000 p.u. | Γωνία = 0.00°
Ζυγός Bus B: Τάση = 1.0000 p.u. | Γωνία = -0.01°

[ ΡΟΕΣ ΓΡΑΜΜΩΝ ]
Γραμμή Line 1: Ροή = 100.00 MW | Φόρτιση = 100.0%

[ ΠΑΡΑΓΩΓΗ ΓΕΝΝΗΤΡΙΩΝ ]
Γεννήτρια Generator A: Ισχύς Εξόδου = 100.0021 MW

Συνολικές Απώλειες Δικτύου: 2.07 kW


In [2]:


# Υπολογισμός ποσοστού φόρτισης για κάθε γραμμή
loading = abs(net.lines_t.p0.iloc[0]) / net.lines.s_nom * 100

# Εύρεση γραμμών με φόρτιση πάνω από 90%
congested_lines = loading[loading > 90]
print("Γραμμές υπό συμφόρηση:")
print(congested_lines)

Γραμμές υπό συμφόρηση:
name
Line 1    100.002066
dtype: float64
